# model-train-eval-toggle-around-sample — ex2: @contextmanager eval_mode: exception-safe eval/no_grad toggle

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `model-train-eval-toggle-around-sample`. Running the final beacon cell reports progress against the `GAN: model.train/eval toggle around sample` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: model.train/eval toggle around sample` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`model-train-eval-toggle-around-sample`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "model-train-eval-toggle-around-sample"
DD_SUBTOPIC = "GAN: model.train/eval toggle around sample"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `eval()` as a context manager — exception-safe restore

Ex1 used the bare `eval() → no_grad → forward → train()` block. The fragile bit: if the forward raises (OOM, NaN, etc.) `train()` never runs and the model is stuck in eval mode for the rest of the loop. The context-manager wrap fixes that with `try/finally`:

```python
from contextlib import contextmanager

@contextmanager
def eval_mode(model):
    was_training = model.training
    model.eval()
    try:
        with t.no_grad():
            yield model
    finally:
        if was_training:
            model.train()
```

**Why capture `was_training`.** If the caller is already in eval mode (e.g. inside a nested `eval_mode`), you must NOT flip to train on exit — restore the prior state. The pre-check is one line and prevents the nested-call footgun.

**Why `finally`.** A raised exception during forward (`NaN`, `CUDA OOM`, asserts) would skip the restore in a try/except. `finally` guarantees the toggle, even when re-raising.

### Exercise 2 — @contextmanager eval_mode: exception-safe eval/no_grad toggle

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `@contextmanager` plus `try/finally` to wrap `model.eval()` + `torch.no_grad()` into a reusable block that guarantees the prior `model.training` state is restored even if the inner block raises.
> Keywords: contextmanager, eval-mode, no_grad, try-finally, exception-safe
> ```

**KCs targeted:** `contextmanager-eval-restore`, `preserve-prior-training-state`

Implement `ex2_eval_mode(model)`, a context manager. Required behavior:

1. Decorate with `@contextlib.contextmanager`.
2. Capture `was_training = model.training` BEFORE entering eval.
3. Call `model.eval()`.
4. Open `with t.no_grad():` and `yield model` from inside it.
5. Wrap the whole `yield` in `try/finally`. In `finally`: if `was_training` was True, call `model.train()`. Otherwise (model was already in eval), leave it in eval.

Critical: the `finally` block must run even if the body of the `with` raises — that's the entire point of the wrapper.

Input: `model` — `nn.Module`.
Yields: `model` (still the same instance, now in eval + no_grad context).

The visualization runs a normal `with ex2_eval_mode(model):` block, then a raising one, and plots BN's `running_mean` state to confirm it's untouched both ways.

In [ ]:
import contextlib

@contextlib.contextmanager
def ex2_eval_mode(model: nn.Module):
    was_training = model.training
    model.eval()
    try:
        with t.no_grad():
            yield model
    finally:
        if was_training:
            model.train()


<details><summary>Solution</summary>

```python
import contextlib

@contextlib.contextmanager
def ex2_eval_mode(model: nn.Module):
    was_training = model.training
    model.eval()
    try:
        with t.no_grad():
            yield model
    finally:
        if was_training:
            model.train()
```

**Pattern: `@contextmanager` + `try/finally`.** The generator-based form of context-manager construction. Code before `yield` runs on `__enter__`; code after `yield` runs on `__exit__`. `try/finally` around the yield is mandatory to guarantee cleanup on exception.

**Why preserve `was_training`.** A caller might already be inside `model.eval()` (e.g. a validation epoch wrapping a metric evaluator). Flipping unconditionally to train on exit would corrupt the outer scope's invariant. Capture-and-restore is the library-grade form.

**`with t.no_grad():` inside, not as a separate context.** Stacking `with eval_mode(m), t.no_grad():` would also work — but embedding no_grad inside the eval_mode wrapper is the single-context API the rest of the training code consumes.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()